# 10a. 베이스라인 데이터 분할 (6:1:1:2)

## 입력: `ST4000DM000_v3.parquet` 전처리 완료 사항
- 상수·정규화·결측률90%↑ 열 삭제, 고장 후 유령 데이터 절단, 완전 중복행 삭제
- 날짜 공백: Case1→개체삭제, Case2→serial_N 분리, Case3→마지막10일삭제 / Forward Fill
- 디코딩: smart_1→Total_Reads, smart_7→seek_error_count/total_seeks, smart_188→Timeout_Total/Timeout_5s
- 온도: smart_190/194 100도↑ → Forward Fill / 삭제: smart_12_raw, smart_240_raw, Timeout_7_5s
- 타겟: D-1~D-10 → failure=1, 고장당일(D-DAY) 삭제

## 분할 전략 (기존 파이프라인과 동일)
- base serial 단위 그룹 층화 분할, seed=42
- `train=0.6 / val_tune=0.1 / val_calib=0.1 / test=0.2`

## 고장 당일 행 추가 (08a 로직)
- val_calib, test에 D-DAY 행을 `ST4000DM000_raw.parquet`에서 디코딩하여 추가

## 출력
- `data/10_baseline_split/train_raw.parquet`
- `data/10_baseline_split/val_tune_raw.parquet`
- `data/10_baseline_split/val_calib_raw.parquet` (고장당일 포함)
- `data/10_baseline_split/test_raw.parquet` (고장당일 포함)

In [1]:
import duckdb, re, time
import pandas as pd
import numpy as np
from pathlib import Path

DATA_ROOT = PROJECT_ROOT / "data2"
INPUT_P   = DATA_ROOT / "01_data_cleaning" / "ST4000DM000_v3.parquet"
RAW_P     = DATA_ROOT / "01_data_cleaning" / "ST4000DM000_raw.parquet"
OUT_DIR   = DATA_ROOT / "10_baseline_split"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_RATIO = {"train_raw": 0.6, "val_tune_raw": 0.1, "val_calib_raw": 0.1, "test_raw": 0.2}
SEED = 42

def strip_suffix(s):
    parts = s.rsplit('_', 1)
    return parts[0] if (len(parts) == 2 and parts[1].isdigit()) else s

print(f"입력(v3): {INPUT_P}")
print(f"입력(raw): {RAW_P}")
print(f"출력: {OUT_DIR}")

입력(v3): C:\Workspace\06_ML_projdect\26_1_COIN\data\01_data_cleaning\ST4000DM000_v3.parquet
입력(raw): C:\Workspace\06_ML_projdect\26_1_COIN\data\01_data_cleaning\ST4000DM000_raw.parquet
출력: C:\Workspace\06_ML_projdect\26_1_COIN\data\10_baseline_split


## 1. EDA

In [2]:
con = duckdb.connect()
ip  = str(INPUT_P)
total_rows  = con.execute(f"SELECT COUNT(*) FROM read_parquet('{ip}')").fetchone()[0]
fail_rows   = con.execute(f"SELECT COUNT(*) FROM read_parquet('{ip}') WHERE failure=1").fetchone()[0]
cols_df     = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{ip}')").fetchdf()
entity_stat = con.execute(f"""
    SELECT COUNT(DISTINCT serial_number) AS total_entities,
           COUNT(DISTINCT CASE WHEN failure=1 THEN serial_number END) AS failed_entities
    FROM read_parquet('{ip}')
""").fetchdf()
con.close()
print(f"총 행: {total_rows:,}  |  Class1: {fail_rows:,} ({fail_rows/total_rows*100:.4f}%)")
print(entity_stat.to_string(index=False))
print(f"\n컬럼({len(cols_df)}): {cols_df['column_name'].tolist()}")

총 행: 79,694,160  |  Class1: 55,965 (0.0702%)
 total_entities  failed_entities
          88374             5667

컬럼(27): ['serial_number', 'date', 'smart_3_raw', 'smart_4_raw', 'smart_5_raw', 'smart_9_raw', 'smart_10_raw', 'smart_183_raw', 'smart_184_raw', 'smart_187_raw', 'smart_189_raw', 'smart_191_raw', 'smart_192_raw', 'smart_193_raw', 'smart_197_raw', 'smart_198_raw', 'smart_199_raw', 'smart_241_raw', 'smart_242_raw', 'total_reads', 'seek_error_count', 'total_seeks', 'failure', 'timeout_total', 'timeout_5s', 'smart_190_raw', 'smart_194_raw']


## 2. 그룹 층화 분할 (6:1:1:2)

In [3]:
start_t = time.time()
con = duckdb.connect()
ip  = str(INPUT_P)

print("[1/3] base serial 별 고장 여부 집계...")
entity_df = con.execute(f"""
    SELECT serial_number,
           REGEXP_REPLACE(serial_number, '_[0-9]+$', '') AS base_serial,
           MAX(failure) AS has_failure
    FROM read_parquet('{ip}')
    GROUP BY serial_number
""").fetchdf()

group_df = (
    entity_df.groupby("base_serial", sort=False)
    .agg(has_failure=("has_failure", "max")).reset_index()
)
print(f"  물리 개체: {len(group_df):,} / 고장: {group_df['has_failure'].sum():,}")

print("[2/3] 층화 분할 수행...")
rng    = np.random.default_rng(SEED)
splits = {}
for label in [1, 0]:
    pool = group_df[group_df["has_failure"] == label]["base_serial"].tolist()
    rng.shuffle(pool)
    n    = len(pool)
    n_tr = round(n * SPLIT_RATIO["train_raw"])
    n_vt = round(n * SPLIT_RATIO["val_tune_raw"])
    n_vc = round(n * SPLIT_RATIO["val_calib_raw"])
    for bs in pool[:n_tr]:                     splits[bs] = "train_raw"
    for bs in pool[n_tr:n_tr+n_vt]:            splits[bs] = "val_tune_raw"
    for bs in pool[n_tr+n_vt:n_tr+n_vt+n_vc]: splits[bs] = "val_calib_raw"
    for bs in pool[n_tr+n_vt+n_vc:]:           splits[bs] = "test_raw"

group_df["split"]  = group_df["base_serial"].map(splits)
entity_df["split"] = entity_df["base_serial"].map(splits)
summary = (
    group_df.groupby("split")
    .agg(entities=("base_serial","count"), failed=("has_failure","sum"))
    .assign(ratio=lambda d: d["failed"]/d["entities"])
)
print("\n[개체 분할 요약]"); print(summary.to_string())

print("\n[3/3] parquet 저장...")
serial_split = dict(zip(entity_df["serial_number"], entity_df["split"]))
for sp in ["train_raw", "val_tune_raw", "val_calib_raw", "test_raw"]:
    serials   = [s for s,v in serial_split.items() if v == sp]
    in_clause = ", ".join(f"'{s}'" for s in serials)
    out_path  = str(OUT_DIR / f"{sp}.parquet")
    con.execute(f"""
        COPY (
            SELECT * FROM read_parquet('{ip}')
            WHERE serial_number IN ({in_clause})
            ORDER BY serial_number, date
        ) TO '{out_path}' (FORMAT PARQUET);
    """)
    rc = con.execute(f"SELECT COUNT(*) FROM read_parquet('{out_path}')").fetchone()[0]
    fc = con.execute(f"SELECT COUNT(*) FROM read_parquet('{out_path}') WHERE failure=1").fetchone()[0]
    print(f"  {sp}: rows={rc:,}  failure={fc:,} ({fc/rc*100:.4f}%)")
con.close()
print(f"\n✅ 분할 완료 ({time.time()-start_t:.1f}초)")

[1/3] base serial 별 고장 여부 집계...
  물리 개체: 36,936 / 고장: 5,667
[2/3] 층화 분할 수행...



[개체 분할 요약]
               entities  failed     ratio
split                                    
test_raw           7387    1133  0.153378
train_raw         22161    3400  0.153423
val_calib_raw      3694     567  0.153492
val_tune_raw       3694     567  0.153492

[3/3] parquet 저장...


  train_raw: rows=47,810,330  failure=33,560 (0.0702%)


  val_tune_raw: rows=7,982,747  failure=5,616 (0.0704%)


  val_calib_raw: rows=7,974,789  failure=5,634 (0.0706%)


  test_raw: rows=15,926,294  failure=11,155 (0.0700%)



✅ 분할 완료 (36.7초)


## 3. 고장 당일 행 추가 (val_calib / test)

08a 로직과 동일:
1. 각 셋의 고장 serial에서 v3 마지막 failure 날짜 + 1일 = D-DAY 계산
2. `raw.parquet`에서 `failure==1` 행 추출 & 비트 디코딩 (NaN→0 후 numpy int64)
3. v3 컬럼 스키마로 정제, **양쪽 모든 컬럼의 dtype을 일치** 시킨 후 concat → 정렬 → 저장

In [4]:
def add_failure_date_rows(split_path, raw_path, v3_cols, output_path):
    print(f"  로드: {split_path}")
    df = pd.read_parquet(split_path)

    df_fail = df[df['failure'] == 1].copy()
    if len(df_fail) == 0:
        print("  고장 개체 없음 - 스킵")
        df.to_parquet(output_path, index=False, compression='zstd')
        return

    failed_serials = df_fail['serial_number'].unique()
    base_serials   = list({strip_suffix(s) for s in failed_serials})

    # 고장일 = v3 마지막 failure 날짜 + 1일
    max_dates = (
        df_fail.groupby('serial_number')['date'].max()
        .reset_index()
        .rename(columns={'date': 'last_date', 'serial_number': 'sn_split'})
    )
    max_dates['base_serial'] = max_dates['sn_split'].apply(strip_suffix)
    max_dates['fail_date']   = pd.to_datetime(max_dates['last_date']) + pd.Timedelta(days=1)
    print(f"  고장 serial: {len(failed_serials):,}  / base: {len(base_serials):,}")

    # raw에서 failure==1 행 로드
    print("  raw 고장 당일 행 로드 & 디코딩...")
    df_raw = pd.read_parquet(raw_path, filters=[('failure', '==', 1)])

    # ── NaN 안전 비트 디코딩: Series→fillna(0)→.values→numpy int64 ──
    # SMART 1 → Total_Reads (하위 32비트)
    if 'smart_1_raw' in df_raw.columns:
        v = pd.to_numeric(df_raw['smart_1_raw'], errors='coerce').fillna(0).values.astype(np.int64)
        df_raw['Total_Reads'] = (v & 0xFFFFFFFF).astype(float)
        df_raw.drop(columns=['smart_1_raw'], inplace=True)

    # SMART 7 → seek_error_count (상위 32비트), total_seeks (하위 32비트)
    if 'smart_7_raw' in df_raw.columns:
        v = pd.to_numeric(df_raw['smart_7_raw'], errors='coerce').fillna(0).values.astype(np.int64)
        df_raw['seek_error_count'] = (v >> 32).astype(float)
        df_raw['total_seeks']      = (v & 0xFFFFFFFF).astype(float)
        df_raw.drop(columns=['smart_7_raw'], inplace=True)

    # SMART 188 → Timeout_Total (하위 16비트), Timeout_5s (중간 16비트)
    if 'smart_188_raw' in df_raw.columns:
        v = pd.to_numeric(df_raw['smart_188_raw'], errors='coerce').fillna(0).values.astype(np.int64)
        df_raw['Timeout_Total'] = (v & 0xFFFF).astype(float)
        df_raw['Timeout_5s']    = ((v >> 16) & 0xFFFF).astype(float)
        df_raw.drop(columns=['smart_188_raw'], inplace=True)

    # 온도 이상치
    for c in ['smart_190_raw', 'smart_194_raw']:
        if c in df_raw.columns:
            df_raw[c] = pd.to_numeric(df_raw[c], errors='coerce')
            df_raw.loc[df_raw[c] >= 100, c] = np.nan

    # 불필요 컬럼 삭제
    df_raw.drop(columns=['smart_12_raw','smart_240_raw','Timeout_7_5s',
                          'model','capacity_bytes'], inplace=True, errors='ignore')

    # base serial 기준 필터
    df_raw = df_raw[df_raw['serial_number'].isin(base_serials)].copy()
    print(f"  디코딩 완료 행: {len(df_raw):,}")

    # 고장 당일 행 구성
    new_rows = []
    for _, mrow in max_dates.iterrows():
        sn_split  = mrow['sn_split']
        base_sn   = mrow['base_serial']
        fail_date = mrow['fail_date']

        raw_row = df_raw[df_raw['serial_number'] == base_sn]
        if len(raw_row) == 0:
            continue
        row = raw_row.iloc[0:1].copy()
        row['serial_number'] = sn_split
        row['date']          = fail_date
        row['failure']       = 1
        new_rows.append(row)

    if not new_rows:
        print("  추가할 고장 당일 행 없음")
        df.to_parquet(output_path, index=False, compression='zstd')
        return

    df_new = pd.concat(new_rows, ignore_index=True)

    # v3 컬럼 스키마에 맞춤
    for c in v3_cols:
        if c not in df_new.columns:
            df_new[c] = np.nan
    df_new = df_new[v3_cols].copy()

    # ── 최종 타입 통일 (pyarrow / pandas dtype mismatch 방지) ──
    for c in v3_cols:
        if c == 'serial_number':
            df_new[c] = df_new[c].astype(df[c].dtype)
        elif c == 'date':
            df_new[c] = pd.to_datetime(df_new[c])
            df[c]     = pd.to_datetime(df[c])
        elif c == 'failure':
            df_new[c] = df_new[c].astype(df[c].dtype)
        else:
            # SMART raw 등 수치형 컬럼
            df_new[c] = pd.to_numeric(df_new[c], errors='coerce').astype(df[c].dtype)

    # 병합 & 정렬
    df_out = pd.concat([df, df_new], ignore_index=True)
    df_out.sort_values(['serial_number','date'], inplace=True)
    df_out.reset_index(drop=True, inplace=True)

    print(f"  원본: {len(df):,} → 추가 후: {len(df_out):,} (추가: {len(df_new):,})")
    df_out.to_parquet(output_path, index=False, compression='zstd')
    print(f"  저장: {output_path}")


print("add_failure_date_rows 함수 정의 완료")

add_failure_date_rows 함수 정의 완료


In [5]:
# v3 컬럼 목록 읽기
con = duckdb.connect()
v3_cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{str(INPUT_P)}')").fetchdf()['column_name'].tolist()
con.close()
print(f"v3 컬럼 ({len(v3_cols)}개): {v3_cols}")

raw_p = str(RAW_P)

print("\n=== val_calib_raw 고장 당일 추가 ===")
add_failure_date_rows(
    split_path  = str(OUT_DIR / "val_calib_raw.parquet"),
    raw_path    = raw_p,
    v3_cols     = v3_cols,
    output_path = str(OUT_DIR / "val_calib_raw.parquet")
)

print("\n=== test_raw 고장 당일 추가 ===")
add_failure_date_rows(
    split_path  = str(OUT_DIR / "test_raw.parquet"),
    raw_path    = raw_p,
    v3_cols     = v3_cols,
    output_path = str(OUT_DIR / "test_raw.parquet")
)

print("\n✅ 고장 당일 행 추가 완료")

v3 컬럼 (27개): ['serial_number', 'date', 'smart_3_raw', 'smart_4_raw', 'smart_5_raw', 'smart_9_raw', 'smart_10_raw', 'smart_183_raw', 'smart_184_raw', 'smart_187_raw', 'smart_189_raw', 'smart_191_raw', 'smart_192_raw', 'smart_193_raw', 'smart_197_raw', 'smart_198_raw', 'smart_199_raw', 'smart_241_raw', 'smart_242_raw', 'total_reads', 'seek_error_count', 'total_seeks', 'failure', 'timeout_total', 'timeout_5s', 'smart_190_raw', 'smart_194_raw']

=== val_calib_raw 고장 당일 추가 ===
  로드: C:\Workspace\06_ML_projdect\26_1_COIN\data\10_baseline_split\val_calib_raw.parquet


  고장 serial: 567  / base: 567
  raw 고장 당일 행 로드 & 디코딩...


  디코딩 완료 행: 567


  원본: 7,974,789 → 추가 후: 7,975,356 (추가: 567)


  저장: C:\Workspace\06_ML_projdect\26_1_COIN\data\10_baseline_split\val_calib_raw.parquet

=== test_raw 고장 당일 추가 ===
  로드: C:\Workspace\06_ML_projdect\26_1_COIN\data\10_baseline_split\test_raw.parquet


  고장 serial: 1,133  / base: 1,133
  raw 고장 당일 행 로드 & 디코딩...


  디코딩 완료 행: 1,133


  원본: 15,926,294 → 추가 후: 15,927,427 (추가: 1,133)


  저장: C:\Workspace\06_ML_projdect\26_1_COIN\data\10_baseline_split\test_raw.parquet



✅ 고장 당일 행 추가 완료


## 4. 누수 검증

In [6]:
con  = duckdb.connect()
data = {}
for sp in ["train_raw", "val_tune_raw", "val_calib_raw", "test_raw"]:
    p  = str(OUT_DIR / f"{sp}.parquet")
    rc = con.execute(f"SELECT COUNT(*) FROM read_parquet('{p}')").fetchone()[0]
    fc = con.execute(f"SELECT COUNT(*) FROM read_parquet('{p}') WHERE failure=1").fetchone()[0]
    print(f"  {sp}: {rc:,} rows, failure={fc:,}")
    data[sp] = set(con.execute(f"SELECT DISTINCT serial_number FROM read_parquet('{p}')").fetchdf()["serial_number"])
con.close()

names = list(data.keys())
print("\n=== base serial 기준 누수 검증 ===")
base = {k: {strip_suffix(s) for s in v} for k,v in data.items()}
for i in range(len(names)):
    for j in range(i+1, len(names)):
        ov = base[names[i]] & base[names[j]]
        print(f"  {names[i]} ∩ {names[j]}: {'✅ 없음' if not ov else f'❌ {len(ov)}건'}")

print("\n✅ 검증 완료")

  train_raw: 47,810,330 rows, failure=33,560
  val_tune_raw: 7,982,747 rows, failure=5,616


  val_calib_raw: 7,975,356 rows, failure=6,201
  test_raw: 15,927,427 rows, failure=12,288



=== base serial 기준 누수 검증 ===
  train_raw ∩ val_tune_raw: ✅ 없음
  train_raw ∩ val_calib_raw: ✅ 없음
  train_raw ∩ test_raw: ✅ 없음
  val_tune_raw ∩ val_calib_raw: ✅ 없음
  val_tune_raw ∩ test_raw: ✅ 없음
  val_calib_raw ∩ test_raw: ✅ 없음

✅ 검증 완료
